#1.SETUP

Connect to github, clone repo if necessary, etc.

In [6]:
from pathlib import Path
import os
import shutil
import subprocess

from google.colab import drive


# ============================================================
# PROJECT SETTINGS
# ============================================================

REPO_NAME = "agentic-ai-network-investigation-team"
GITHUB_USER = "icarovazquez"
BRANCH = "main"

DRIVE_MOUNT = Path("/content/drive")
PROJECTS_DIR = DRIVE_MOUNT / "MyDrive" / "Colab Notebooks"
PROJECT_ROOT = PROJECTS_DIR / REPO_NAME

# Store the persistent key in Google Drive.
DRIVE_SSH_DIR = DRIVE_MOUNT / "MyDrive" / ".ssh_colab"
DRIVE_PRIVATE_KEY = DRIVE_SSH_DIR / "id_ed25519"
DRIVE_PUBLIC_KEY = DRIVE_SSH_DIR / "id_ed25519.pub"

# Runtime SSH directory. Colab resets this when the runtime restarts.
RUNTIME_SSH_DIR = Path.home() / ".ssh"
RUNTIME_PRIVATE_KEY = RUNTIME_SSH_DIR / "id_ed25519"
RUNTIME_PUBLIC_KEY = RUNTIME_SSH_DIR / "id_ed25519.pub"
SSH_CONFIG = RUNTIME_SSH_DIR / "config"
KNOWN_HOSTS = RUNTIME_SSH_DIR / "known_hosts"

SSH_REPO_URL = (
    f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
)


# ============================================================
# COMMAND HELPER
# ============================================================

def run_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess:
    """Run a shell command and display its output."""

    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"{' '.join(command)}"
        )

    return result


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("✓ Google Drive is already mounted.")


# ============================================================
# 2. CREATE A PERSISTENT SSH KEY IF NEEDED
# ============================================================

DRIVE_SSH_DIR.mkdir(parents=True, exist_ok=True)

new_key_created = False

if not DRIVE_PRIVATE_KEY.exists():
    print("Creating a persistent SSH key in Google Drive...")

    run_command(
        [
            "ssh-keygen",
            "-t",
            "ed25519",
            "-C",
            "icarovazquez-colab",
            "-f",
            str(DRIVE_PRIVATE_KEY),
            "-N",
            "",
        ]
    )

    new_key_created = True
    print("✓ Persistent SSH key created.")
else:
    print("✓ Persistent SSH key already exists in Google Drive.")


# ============================================================
# 3. COPY THE KEY INTO THE COLAB RUNTIME
# ============================================================

RUNTIME_SSH_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_PRIVATE_KEY, RUNTIME_PRIVATE_KEY)
shutil.copy2(DRIVE_PUBLIC_KEY, RUNTIME_PUBLIC_KEY)

os.chmod(RUNTIME_SSH_DIR, 0o700)
os.chmod(RUNTIME_PRIVATE_KEY, 0o600)
os.chmod(RUNTIME_PUBLIC_KEY, 0o644)

print("✓ SSH key copied into the Colab runtime.")


# ============================================================
# 4. CREATE SSH CONFIGURATION
# ============================================================

SSH_CONFIG.write_text(
    f"""Host github.com
    HostName github.com
    User git
    IdentityFile {RUNTIME_PRIVATE_KEY}
    IdentitiesOnly yes
"""
)

os.chmod(SSH_CONFIG, 0o600)

print("✓ SSH configuration created.")


# ============================================================
# 5. ADD GITHUB TO KNOWN_HOSTS
# ============================================================

keyscan = subprocess.run(
    ["ssh-keyscan", "-t", "ed25519", "github.com"],
    text=True,
    capture_output=True,
    check=True,
)

KNOWN_HOSTS.write_text(keyscan.stdout)
os.chmod(KNOWN_HOSTS, 0o644)

print("✓ GitHub added to known_hosts.")


# ============================================================
# FIRST-RUN GITHUB REGISTRATION
# ============================================================

if new_key_created:
    print("\n" + "=" * 70)
    print("ONE-TIME GITHUB SETUP REQUIRED")
    print("=" * 70)
    print(
        "Copy the public key below and add it at:\n"
        "GitHub → Settings → SSH and GPG keys → New SSH key\n"
    )

    print(DRIVE_PUBLIC_KEY.read_text().strip())

    print(
        "\nAfter adding the key to GitHub, rerun this cell."
    )

    raise SystemExit(
        "SSH key created. Add the public key to GitHub, then rerun."
    )


# ============================================================
# TEST GITHUB AUTHENTICATION
# ============================================================

ssh_test = run_command(
    [
        "ssh",
        "-o",
        "StrictHostKeyChecking=yes",
        "-T",
        "git@github.com",
    ],
    check=False,
)

# GitHub returns exit code 1 even when authentication succeeds.
ssh_output = (
    ssh_test.stdout + "\n" + ssh_test.stderr
).lower()

if "successfully authenticated" not in ssh_output:
    raise RuntimeError(
        "GitHub SSH authentication failed.\n"
        "Confirm that the displayed public key was added to your "
        "GitHub account under SSH and GPG keys."
    )

print("✓ GitHub SSH authentication succeeded.")


# ============================================================
# 6. CLONE OR UPDATE THE REPOSITORY
# ============================================================

PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    print(f"Cloning repository into:\n{PROJECT_ROOT}")

    run_command(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            SSH_REPO_URL,
            str(PROJECT_ROOT),
        ]
    )

elif not (PROJECT_ROOT / ".git").exists():
    raise RuntimeError(
        f"{PROJECT_ROOT} exists but is not a Git repository."
    )

else:
    print(f"✓ Repository already exists at:\n{PROJECT_ROOT}")

    # Ensure the remote uses SSH rather than HTTPS.
    run_command(
        [
            "git",
            "remote",
            "set-url",
            "origin",
            SSH_REPO_URL,
        ],
        cwd=PROJECT_ROOT,
    )

    status = run_command(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
    )

    if status.stdout.strip():
        print(
            "\nLocal changes detected. Git pull was skipped so that "
            "uncommitted work is not overwritten."
        )
    else:
        print(f"Pulling the latest origin/{BRANCH} changes...")

        run_command(
            [
                "git",
                "pull",
                "--ff-only",
                "origin",
                BRANCH,
            ],
            cwd=PROJECT_ROOT,
        )


# ============================================================
# ENTER THE PROJECT DIRECTORY
# ============================================================

os.chdir(PROJECT_ROOT)

print("\n" + "=" * 70)
print("PROJECT INITIALIZATION COMPLETE")
print("=" * 70)
print(f"Project root: {PROJECT_ROOT}")

run_command(["git", "remote", "-v"], cwd=PROJECT_ROOT)
run_command(["git", "status", "--short", "--branch"], cwd=PROJECT_ROOT)

✓ Google Drive is already mounted.
✓ Persistent SSH key already exists in Google Drive.
✓ SSH key copied into the Colab runtime.
✓ SSH configuration created.
✓ GitHub added to known_hosts.
Hi icarovazquez! You've successfully authenticated, but GitHub does not provide shell access.
✓ GitHub SSH authentication succeeded.
✓ Repository already exists at:
/content/drive/MyDrive/Colab Notebooks/agentic-ai-network-investigation-team
M "notebooks/Agentic AI Network Investigation Team.ipynb"

Local changes detected. Git pull was skipped so that uncommitted work is not overwritten.

PROJECT INITIALIZATION COMPLETE
Project root: /content/drive/MyDrive/Colab Notebooks/agentic-ai-network-investigation-team
origin	git@github.com:icarovazquez/agentic-ai-network-investigation-team.git (fetch)
origin	git@github.com:icarovazquez/agentic-ai-network-investigation-team.git (push)
## main...origin/main
 M "notebooks/Agentic AI Network Investigation Team.ipynb"


CompletedProcess(args=['git', 'status', '--short', '--branch'], returncode=0, stdout='## main...origin/main\n M "notebooks/Agentic AI Network Investigation Team.ipynb"\n', stderr='')